In [8]:
import os
import re
import math
from collections import Counter
import pandas as pd

# Đường dẫn gốc tới corpus
root_path = r"D:\Research 2\archive\Train_Full"

max_files_per_folder = 5  # bạn có thể tăng lên 10 nếu máy ổn

# Một số dòng/nhãn kiểu công thức, giá cả, nguyên liệu... không có ích cho scope project
# Có thể mở rộng thêm sau khi xem dữ liệu thật nhiều hơn
bad_line_starts = {
    "nguyên liệu",
    "thực hiện",
    "giá tham khảo",
}

def clean_line_for_project(line):
    """
    Clean một dòng theo hướng có lợi cho project:
    - giữ tiếng Việt có dấu
    - bỏ số
    - bỏ dấu câu / ký hiệu
    - giữ khoảng trắng để bảo toàn syllable structure
    """
    line = line.strip().lower()

    # Bỏ các dòng rỗng
    if not line:
        return ""

    # Bỏ một số dòng tiêu đề kỹ thuật không có ích cho lexical decision stimuli
    if line in bad_line_starts or any(line.startswith(x + ":") for x in bad_line_starts):
        return ""

    # Bỏ số
    line = re.sub(r"\d+", " ", line)

    # Đổi dấu gạch nối thành khoảng trắng để tránh dính token
    line = line.replace("-", " ")

    # Giữ chữ cái, số gạch dưới mặc định của \w, khoảng trắng và chữ tiếng Việt có dấu
    # Mục tiêu ở đây là bỏ dấu câu/ký hiệu, không đụng vào chữ tiếng Việt
    line = re.sub(
        r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]",
        " ",
        line
    )

    # Bỏ dấu gạch dưới nếu có
    line = line.replace("_", " ")

    # Chuẩn hóa nhiều khoảng trắng liên tiếp
    line = re.sub(r"\s+", " ", line).strip()

    return line

def generate_ngrams(tokens, n):
    """
    Tạo n-gram liên tiếp từ danh sách syllables.
    Ví dụ:
    ['học', 'sinh', 'giỏi'] với n=2
    -> ['học sinh', 'sinh giỏi']
    """
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

all_unigrams = []
all_bigrams = []
all_trigrams = []

file_count = 0

for root, dirs, files in os.walk(root_path):

    file_counter = 0  # reset cho mỗi folder

    for file in files:
        if file.endswith(".txt"):

            if file_counter >= max_files_per_folder:
                break

            file_path = os.path.join(root, file)
            file_counter += 1
            file_count += 1

            # đọc file với fallback encoding để tránh crash
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
            except UnicodeDecodeError:
                try:
                    with open(file_path, "r", encoding="utf-16") as f:
                        lines = f.readlines()
                except:
                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        lines = f.readlines()

            cleaned_lines = []

            for line in lines:
                clean_line = clean_line_for_project(line)
                if clean_line:
                    cleaned_lines.append(clean_line)

            text = " ".join(cleaned_lines)
            tokens = text.split()

            if not tokens:
                continue

            all_unigrams.extend(tokens)
            all_bigrams.extend(generate_ngrams(tokens, 2))
            all_trigrams.extend(generate_ngrams(tokens, 3))

print(f"So file da doc: {file_count}")
print(f"So syllable tokens: {len(all_unigrams)}")
print(f"So bigrams: {len(all_bigrams)}")
print(f"So trigrams: {len(all_trigrams)}")

# Tính frequency
freq_uni = Counter(all_unigrams)
freq_bi = Counter(all_bigrams)
freq_tri = Counter(all_trigrams)

# Đổi sang DataFrame để dễ nhìn và lọc
df_uni = pd.DataFrame(freq_uni.items(), columns=["lexical_item", "frequency"])
df_bi = pd.DataFrame(freq_bi.items(), columns=["lexical_item", "frequency"])
df_tri = pd.DataFrame(freq_tri.items(), columns=["lexical_item", "frequency"])

# Gắn số syllables
df_uni["syllable_length"] = 1
df_bi["syllable_length"] = 2
df_tri["syllable_length"] = 3

# Log frequency để chuẩn bị cho analysis sau này
df_uni["log_frequency"] = df_uni["frequency"].apply(lambda x: math.log(x))
df_bi["log_frequency"] = df_bi["frequency"].apply(lambda x: math.log(x))
df_tri["log_frequency"] = df_tri["frequency"].apply(lambda x: math.log(x))

# Gộp lại thành một bảng chung
df_all = pd.concat([df_uni, df_bi, df_tri], ignore_index=True)

# Sắp xếp giảm dần theo frequency
df_all = df_all.sort_values(by=["syllable_length", "frequency"], ascending=[True, False])

# Lưu output
output_dir = os.path.join(root_path, "_processed")
os.makedirs(output_dir, exist_ok=True)

df_uni.to_csv(os.path.join(output_dir, "unigrams_frequency.csv"), index=False, encoding="utf-8-sig")
df_bi.to_csv(os.path.join(output_dir, "bigrams_frequency.csv"), index=False, encoding="utf-8-sig")
df_tri.to_csv(os.path.join(output_dir, "trigrams_frequency.csv"), index=False, encoding="utf-8-sig")
df_all.to_csv(os.path.join(output_dir, "all_1to3gram_frequency.csv"), index=False, encoding="utf-8-sig")

print("\nTop 20 unigrams:")
print(df_uni.sort_values("frequency", ascending=False).head(20).to_string(index=False))

print("\nTop 20 bigrams:")
print(df_bi.sort_values("frequency", ascending=False).head(20).to_string(index=False))

print("\nTop 20 trigrams:")
print(df_tri.sort_values("frequency", ascending=False).head(20).to_string(index=False))

print(f"\nDa luu file vao: {output_dir}")

So file da doc: 40
So syllable tokens: 20721
So bigrams: 20681
So trigrams: 20641

Top 20 unigrams:
lexical_item  frequency  syllable_length  log_frequency
          và        261                1       5.564520
          có        230                1       5.438079
         của        226                1       5.420535
         cho        185                1       5.220356
         các        181                1       5.198497
       không        174                1       5.159055
       trong        157                1       5.056246
          là        155                1       5.043425
         với        154                1       5.036953
        được        153                1       5.030438
       người        143                1       4.962845
       những        142                1       4.955827
         giá        125                1       4.828314
         một        119                1       4.779123
          đã        112                1       4.718499
    